# SystemTopology `add` Alias

Regression test for the deprecated `SystemTopology.add` alias
([#530](https://github.com/sogno-platform/dpsim/issues/530)): it must keep
working for scripts written before the rename to `add_component`/`add_node`,
emit a `DeprecationWarning`, dispatch nodes and components to the right
method, accept lists and tuples, and reject invalid input with a `TypeError`.

In [ ]:
import warnings

import dpsimpy

In [ ]:
system = dpsimpy.SystemTopology(50)

n1 = dpsimpy.emt.SimNode("n1")
r1 = dpsimpy.emt.ph1.Resistor("r1")
r1.set_parameters(R=1)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    system.add(r1)
    system.add(n1)

assert len(caught) == 2
assert all(issubclass(w.category, DeprecationWarning) for w in caught)
assert [c.name() for c in system.components] == ["r1"]
assert any(n.name() == "n1" for n in system.nodes)
print("single component and node dispatch: ok")

In [ ]:
r2 = dpsimpy.emt.ph1.Resistor("r2")
r2.set_parameters(R=2)
r3 = dpsimpy.emt.ph1.Resistor("r3")
r3.set_parameters(R=3)
n2 = dpsimpy.emt.SimNode("n2")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    system.add([r2, n2])
    system.add((r3,))

assert len(caught) == 2
assert sorted(c.name() for c in system.components) == ["r1", "r2", "r3"]
assert any(n.name() == "n2" for n in system.nodes)
print("list and tuple dispatch: ok")

In [ ]:
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        system.add(42)
except TypeError:
    print("invalid input raises TypeError: ok")
else:
    raise AssertionError("add(42) must raise TypeError")

with warnings.catch_warnings():
    warnings.simplefilter("error", DeprecationWarning)
    try:
        system.add(r1)
    except DeprecationWarning:
        print("warning-as-error propagates: ok")
    else:
        raise AssertionError("expected DeprecationWarning to be raised")

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    r4 = dpsimpy.emt.ph1.Resistor("r4")
    r4.set_parameters(R=4)
    system.add_component(r4)
    system.add_node(dpsimpy.emt.SimNode("n3"))

assert not any(issubclass(w.category, DeprecationWarning) for w in caught)
print("add_component and add_node stay silent: ok")